```
pip install uv
uv vnev
uv pip install ipykernel pandas requests mecab gensim
```

In [1]:
import requests
import pandas as pd
import json

In [2]:
# OpenSearchの接続情報
BASE_URL = "http://localhost:9200"
INDEX_NAME = "ndl-bibliography"
HEADERS = {"Content-Type": "application/json"}

In [3]:
print(requests.get(f"{BASE_URL}/_cat/plugins").text)

1db9e3896264 analysis-kuromoji                    3.0.0
1db9e3896264 opensearch-alerting                  3.0.0.0
1db9e3896264 opensearch-anomaly-detection         3.0.0.0
1db9e3896264 opensearch-asynchronous-search       3.0.0.0
1db9e3896264 opensearch-cross-cluster-replication 3.0.0.0
1db9e3896264 opensearch-custom-codecs             3.0.0.0
1db9e3896264 opensearch-flow-framework            3.0.0.0
1db9e3896264 opensearch-geospatial                3.0.0.0
1db9e3896264 opensearch-index-management          3.0.0.0
1db9e3896264 opensearch-job-scheduler             3.0.0.0
1db9e3896264 opensearch-knn                       3.0.0.0
1db9e3896264 opensearch-ltr                       3.0.0.0
1db9e3896264 opensearch-ml                        3.0.0.0
1db9e3896264 opensearch-neural-search             3.0.0.0
1db9e3896264 opensearch-notifications             3.0.0.0
1db9e3896264 opensearch-notifications-core        3.0.0.0
1db9e3896264 opensearch-observability             3.0.0.0
1db9e3896264 ope

## schema(mappingの定義)

In [ ]:
mapping = {
    "settings": {
        "analysis": {
            "tokenizer": {"kuromoji_tokenizer": {"type": "kuromoji_tokenizer"}},
            "analyzer": {
                "kuromoji_analyzer": {
                    "type": "custom",
                    "tokenizer": "kuromoji_tokenizer",
                }
            },
        }
    },
    "mappings": {
        "properties": {
            "永続的識別子": {"type": "keyword"},
            "タイトル": {"type": "text", "analyzer": "kuromoji_analyzer"},
            "巻次又は部編番号": {"type": "keyword"},
            "シリーズタイトル": {"type": "text", "analyzer": "kuromoji_analyzer"},
            "版表示": {"type": "keyword"},
            "著者": {"type": "text", "analyzer": "kuromoji_analyzer"},
            "出版者": {"type": "keyword"},
            "出版日": {"type": "keyword"},
            "出版日（W3CDTF）": {
                "type": "date",
                "format": "yyyy||yyyy-MM||yyyy-MM-dd||strict_date_optional_time||epoch_millis",
            },
            "識別子（ISBN）": {"type": "keyword"},
            "容量・大きさ": {"type": "text"},
            "分類（NDC）": {"type": "keyword"},
            "分類（NDC8）": {"type": "keyword"},
            "分類（NDC9）": {"type": "keyword"},
            "分類（NDLC）": {"type": "keyword"},
            "件名（NDLSH）": {"type": "text", "analyzer": "kuromoji_analyzer"},
            "公開範囲": {"type": "keyword"},
        }
    },
}

In [ ]:
res = requests.put(
    f"{BASE_URL}/{INDEX_NAME}", headers=HEADERS, data=json.dumps(mapping)
)
res.json()

In [ ]:
# requests.delete(f"{BASE_URL}/{INDEX_NAME}")

## Bulk APIを用いたfeed

In [ ]:
# df = pd.read_csv("../head.tsv", sep="\t")
df = pd.concat(
    [
        pd.read_csv("../data/dataset_202405_t_kannai_01.tsv", sep="\t"),
        pd.read_csv("../data/dataset_202405_t_kannai_02.tsv", sep="\t"),
        pd.read_csv("../data/dataset_202405_t_kannai_03.tsv", sep="\t"),
    ]
)
df.head(3)

In [ ]:
actions = df["永続的識別子"].map(
    lambda x: json.dumps(
        {
            "index": {
                "_index": INDEX_NAME,
                "_id": x.translate(str.maketrans({":": "-", "/": "-"})),
            }
        }
    )
)
data_jsons = df.to_json(orient="records", lines=True, force_ascii=False).splitlines()

jsonls = [
    action + "\n" + data_json + "\n" for action, data_json in zip(actions, data_jsons)
]

In [ ]:
# 10000件ずつ送信
chunk_size = 10000
for i in range(0, len(jsonls), chunk_size):
    print(f"Sending chunk {i} ~ {i + chunk_size}")
    jsonls_chunk = jsonls[i : i + chunk_size]
    print(jsonls_chunk[:3])  

    res = requests.post(
        f"{BASE_URL}/{INDEX_NAME}/_bulk", headers=HEADERS, data="".join(jsonls_chunk)
    )
    res.raise_for_status()

In [ ]:
# # 削除
# requests.post(f"{BASE_URL}/{INDEX_NAME}/_delete_by_query", headers=HEADERS, data=json.dumps(
# {
#   "query": {
#     "match_all": {}
#   }
# })).json()

In [ ]:
count_res = requests.get(f"{BASE_URL}/{INDEX_NAME}/_count", headers=HEADERS)
print("=== _count API ===")
print(f"総ドキュメント数: {count_res.json()['count']}")
print(count_res.json())

In [ ]:

requests.post(f"{BASE_URL}/{INDEX_NAME}/_search", headers=HEADERS, data=json.dumps(
{
  "query": {
    "match_all": {}
  }
})).json()

## 検索

In [51]:
query = {"query": {"match": {"著者": "東野圭吾"}}}

res = requests.post(
    f"{BASE_URL}/{INDEX_NAME}/_search", data=json.dumps(query), headers=HEADERS
)
res.json()

{'took': 140,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 35, 'relation': 'eq'},
  'max_score': 10.687419,
  'hits': [{'_index': 'ndl-bibliography',
    '_id': 'info-ndljp-pid-12478369',
    '_score': 10.687419,
    '_source': {'永続的識別子': 'info:ndljp/pid/12478369',
     'タイトル': '卒業 : 雪月花殺人ゲーム',
     '巻次又は部編番号': None,
     'シリーズタイトル': None,
     '版表示': None,
     '著者': '東野圭吾 著',
     '出版者': '講談社',
     '出版日': '1986.5',
     '出版日（W3CDTF）': '1986',
     '識別子（ISBN）': '4-06-202728-3',
     '容量・大きさ': '319p ; 20cm',
     '分類（NDC）': None,
     '分類（NDC8）': 913.6,
     '分類（NDC9）': None,
     '分類（NDLC）': 'KH139',
     '件名（NDLSH）': None,
     '公開範囲': '国立国会図書館内限定公開'}},
   {'_index': 'ndl-bibliography',
    '_id': 'info-ndljp-pid-12478360',
    '_score': 10.687419,
    '_source': {'永続的識別子': 'info:ndljp/pid/12478360',
     'タイトル': '学生街の殺人',
     '巻次又は部編番号': None,
     'シリーズタイトル': None,
     '版表示': None,
     '著者': '東野圭吾 著',
  

In [ ]:
requests.get(f"{BASE_URL}/{INDEX_NAME}/_search?q=*:*").json()

## featuresetの定義と利用


In [ ]:
data = {"mappings": {"properties": {}}}
res = requests.put(f"{BASE_URL}/.ltrstore", headers=HEADERS, data=json.dumps(data))
print(res.json())

In [ ]:
data = {
    "featureset": {
        "features": [
            {
                "name": "title_match",
                "params": ["keywords"],
                "template": {"match": {"タイトル": "{{keywords}}"}},
            },
            {
                "name": "author_match",
                "params": ["keywords"],
                "template": {"match": {"著者": "{{keywords}}"}},
            },
            {
                "name": "publisher_match",
                "params": ["keywords"],
                "template": {"match": {"出版者": "{{keywords}}"}},
            },
        ]
    }
}

requests.post(
    f"{BASE_URL}/_ltr/_featureset/sample_featureset",
    data=json.dumps(data),
    headers=HEADERS,
).json()

In [ ]:
requests.get(f"{BASE_URL}/_ltr/_featureset").json()

## 特徴量をエンジンから計算

In [ ]:
query = {
    "query": {
        "bool": {
            "filter": [
                {
                    "terms": {
                        "_id": [
                            "info-ndljp-pid-12478369",
                            "info-ndljp-pid-13256368",
                        ]
                    }
                },
                {
                    "sltr": {
                        "_name": "logged_featureset",
                        "featureset": "sample_featureset",
                        "params": {"keywords": "東野圭吾"},
                    }
                },
            ]
        }
    },
    "ext": {
        "ltr_log": {
            "log_specs": {"name": "log_entry1", "named_query": "logged_featureset"}
        }
    },
}

res = requests.post(
    f"{BASE_URL}/{INDEX_NAME}/_search", data=json.dumps(query), headers=HEADERS
)
res.json()

In [ ]:
query = {
    "size": 10,
    "query": {
        "bool": {
            "must": [{"match": {"著者": {"query": "東野圭吾", "operator": "and"}}}],
            "filter": [
                {
                    "sltr": {
                        "_name": "logged_featureset",
                        "featureset": "sample_featureset",
                        "params": {"keywords": "東野圭吾"},
                    }
                }
            ],
        }
    },
    "ext": {
        "ltr_log": {
            "log_specs": {"name": "log_entry1", "named_query": "logged_featureset"}
        }
    },
}

res = requests.post(
    f"{BASE_URL}/{INDEX_NAME}/_search", data=json.dumps(query), headers=HEADERS
)
res.json()

## モデルのデプロイ
- いったん雑線形モデルから

In [6]:
# "model_type" ではなく "type" を使う必要があります

query = {
    "model": {
        "name": "sample_linear_model",
        "model": {
            "type": "model/linear",
            "definition": json.dumps(
                {"title_match": 0.2, "author_match": 0.1, "publisher_match": 0.1}
            ),
        },
    }
}

endpoint = f"{BASE_URL}/_ltr/_featureset/sample_featureset/_createmodel"
res = requests.post(endpoint, data=json.dumps(query), headers=HEADERS)
res.json()

{'error': {'root_cause': [{'type': 'illegal_argument_exception',
    'reason': 'Element of type [model] are not updatable, please create a new one instead.'}],
  'type': 'illegal_argument_exception',
  'reason': 'Element of type [model] are not updatable, please create a new one instead.',
  'suppressed': [{'type': 'version_conflict_engine_exception',
    'reason': '[model-sample_linear_model]: version conflict, document already exists (current version [3])',
    'index': '.ltrstore',
    'shard': '0',
    'index_uuid': 'uvJyswt-TjCTS2EgYm1plw'}]},
 'status': 405}

In [ ]:
# delete_endpoint = f"{BASE_URL}/_ltr/_model/sample_linear_model"
# res = requests.delete(delete_endpoint, headers=HEADERS)
# res.json()

In [ ]:
requests.get(f"{BASE_URL}/_ltr/_model").json()

In [ ]:
# ベクトル化関数（最小化版）
import numpy as np

def vectorize_text(text, model, vector_size):
    """テキストをWord2VecのAvgPoolingでベクトル化"""
    if not text:
        return np.zeros(vector_size)
    
    valid_vectors = []
    for word in text:
        if word in model.wv:
            valid_vectors.append(model.wv[word])
    
    if valid_vectors:
        return np.mean(valid_vectors, axis=0)
    else:
        return np.zeros(vector_size)

## ベクトル検索の実装

In [4]:
import MeCab
import numpy as np
import re
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd


In [5]:

# データ読み込み
# df = pd.read_csv("../head.tsv", sep="\t")
df = pd.concat(
    [
        pd.read_csv("../data/dataset_202405_t_kannai_01.tsv", sep="\t"),
        pd.read_csv("../data/dataset_202405_t_kannai_02.tsv", sep="\t"),
        pd.read_csv("../data/dataset_202405_t_kannai_03.tsv", sep="\t"),
    ]
)


/tmp/ipykernel_858/3411647010.py:5: DtypeWarning: Columns (11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv("../data/dataset_202405_t_kannai_01.tsv", sep="\t"),
/tmp/ipykernel_858/3411647010.py:6: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv("../data/dataset_202405_t_kannai_02.tsv", sep="\t"),
/tmp/ipykernel_858/3411647010.py:7: DtypeWarning: Columns (11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv("../data/dataset_202405_t_kannai_03.tsv", sep="\t"),


In [6]:
# 日本語テキスト前処理（最小化版）
import re
import pandas as pd
import MeCab

def preprocess_japanese_text(text):
    """日本語テキストの前処理"""
    if pd.isna(text) or text == "":
        return []
    
    tagger = MeCab.Tagger("-r /etc/mecabrc -Owakati")
    text_cleaned = re.sub(r'[^\w\s\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FAF]', ' ', str(text))
    result = tagger.parse(text_cleaned).strip()
    words = result.split()
    return [word for word in words if len(word) > 1]

# テスト実行
test_text = "吾輩は猫である。名前はまだ無い。"
print(f"テスト入力: {test_text}")
result = preprocess_japanese_text(test_text)
print(f"分かち書き結果: {result}")

テスト入力: 吾輩は猫である。名前はまだ無い。
分かち書き結果: ['吾輩', 'ある', '名前', 'まだ', '無い']


In [ ]:
# データの準備とWord2Vecモデルの訓練
print("=== テキストデータの準備 ===")

# タイトルと著者のテキストを結合
df['combined_text'] = df['タイトル'].fillna('') + ' ' + df['著者'].fillna('')

# 各行のテキストを形態素解析
print("形態素解析実行中...")
sentences = []
for idx, text in enumerate(df['combined_text']):
    words = preprocess_japanese_text(text)
    if words:  # 空でない場合のみ追加
        sentences.append(words)

print("サンプル文書:", sentences[:3])

=== テキストデータの準備 ===
形態素解析実行中...


In [9]:
# Word2Vecモデルの訓練
print("\n=== Word2Vecモデル訓練 ===")

# Word2Vecのパラメータ
vector_size = 128  # ベクトルの次元数
window = 5        # 文脈窓のサイズ
min_count = 2     # 最小出現回数
workers = 4       # 並列処理数

# モデル訓練
print("Word2Vecモデル訓練中...")
model = Word2Vec(
    sentences=sentences,
    vector_size=vector_size,
    window=window,
    min_count=min_count,
    workers=workers,
    epochs=10
)

print(f"語彙数: {len(model.wv.key_to_index)}")
print("訓練完了")

# モデルを保存
model.save("../data/word2vec_model.model")
print(f"モデルを保存しました")

# モデルテスト
test_words = ["東野", "圭吾", "小説", "推理"]
for word in test_words:
    if word in model.wv:
        similar_words = model.wv.most_similar(word, topn=5)
        print(f"'{word}'に類似する語: {similar_words}")


=== Word2Vecモデル訓練 ===
Word2Vecモデル訓練中...
語彙数: 61840
訓練完了
モデルを保存しました
'東野'に類似する語: [('愛宕', 0.8340176939964294), ('旅芸人', 0.8083248138427734), ('峯岸', 0.8016675114631653), ('注疏', 0.7975339889526367), ('黙阿弥', 0.7961704134941101)]
'圭吾'に類似する語: [('島民', 0.6947019100189209), ('長登', 0.6945280432701111), ('写さ', 0.6931813955307007), ('覇王', 0.6923402547836304), ('英彦山', 0.6895477175712585)]
'小説'に類似する語: [('ネビロス', 0.7165817618370056), ('妖殺', 0.714601457118988), ('オカルトロマンス', 0.6776121258735657), ('長編', 0.6723307967185974), ('推理', 0.6680236458778381)]
'推理'に類似する語: [('ミステリー', 0.8390397429466248), ('本格', 0.8008485436439514), ('サスペンス', 0.7973572015762329), ('長編', 0.7829282283782959), ('ネビロス', 0.7580593228340149)]


In [10]:
# AvgPoolingベースのベクトル化関数
def text_to_vector(text, model, vector_size=128):
    """
    テキストをWord2VecのAvgPoolingでベクトル化する関数。
    
    - 入力: 
        - text: ベクトル化対象のテキスト（日本語）。
        - model: Word2Vecモデル。
        - vector_size: ベクトルの次元数（デフォルト128）。
    - 処理:
        - テキストを形態素解析して単語リストに変換。
        - Word2Vecモデルに存在する単語のベクトルを取得。
        - 単語ベクトルの平均を計算して文書ベクトルを生成。
    - 出力:
        - 文書ベクトル（numpy配列）。
    """
    words = preprocess_japanese_text(text)
    if not words:
        return np.zeros(vector_size)
    
    # 語彙に存在する単語のベクトルを取得
    word_vectors = []
    for word in words:
        if word in model.wv:
            word_vectors.append(model.wv[word])
    
    if not word_vectors:
        return np.zeros(vector_size)
    
    # 平均プーリング
    return np.mean(word_vectors, axis=0)

# テスト実行
print("\n=== ベクトル化テスト ===")
test_docs = [
    "東野圭吾 容疑者Xの献身",
    "村上春樹 ノルウェイの森", 
    "夏目漱石 吾輩は猫である"
]

vectors = []
for doc in test_docs:
    vec = text_to_vector(doc, model, vector_size)
    vectors.append(vec)
    print(f"'{doc}' -> ベクトル形状: {vec.shape}, 最初の5要素: {vec[:5]}")

# ベクトル間の類似度計算
vectors = np.array(vectors)
similarity_matrix = cosine_similarity(vectors)
print(f"\n類似度行列:\n{similarity_matrix}")


=== ベクトル化テスト ===
'東野圭吾 容疑者Xの献身' -> ベクトル形状: (128,), 最初の5要素: [0.07690488 0.02128709 0.06006924 0.04977345 0.15734369]
'村上春樹 ノルウェイの森' -> ベクトル形状: (128,), 最初の5要素: [ 0.37892914 -0.8054347   0.3465003   0.2697939  -0.1579126 ]
'夏目漱石 吾輩は猫である' -> ベクトル形状: (128,), 最初の5要素: [ 0.70008427 -0.55556524  0.864253    0.6139221   0.5229395 ]

類似度行列:
[[1.0000001  0.3563305  0.29085067]
 [0.3563305  0.99999994 0.27281594]
 [0.29085067 0.27281594 0.9999999 ]]


## OpenSearchベクトル検索の実装
先ほど作ったKW用indexとは異なるindexを作成してそちらで検索などする

In [11]:
# ベクトル検索用インデックスの作成
VECTOR_INDEX_NAME = "ndl-bibliography-vector"

query = {
  "training_index": VECTOR_INDEX_NAME,
  "training_field": "title_author_vector",
  "dimension": vector_size,
  "description": "My model description",
  "method": {
    "name": "ivf",
    "engine": "faiss",
    "parameters": {
      "encoder": {
        "name": "pq",
        "parameters": {
          "code_size": 2,
          "m": 2
        }
      }
    }
  }
}

# knnのindexをビルド
res = requests.post(
    f"{BASE_URL}/_plugins/_knn/models/title_author_index/_train",
    headers=HEADERS,
    data=json.dumps(query)
)
res.json()

{'error': {'root_cause': [{'type': 'action_request_validation_exception',
    'reason': 'Validation Failed: 1: Model with id="title_author_index" already exists;'}],
  'type': 'action_request_validation_exception',
  'reason': 'Validation Failed: 1: Model with id="title_author_index" already exists;'},
 'status': 400}

In [12]:
requests.delete(f"{BASE_URL}/{VECTOR_INDEX_NAME}")

<Response [200]>

In [13]:
#  /_plugins/_knn/models/my-model?filter_path=state&pretty

res = requests.get(
    f"{BASE_URL}/_plugins/_knn/models/title_author_index",
)
res.json()

{'model_id': 'title_author_index',
 'model_blob': 'SXdQUYAAAAAAAAAAAAAAAAAAEAAAAAAAAAAQAAAAAAABAQAAAAQAAAAAAAAAAQAAAAAAAABJeEYygAAAAAQAAAAAAAAAAAAQAAAAAAAAABAAAAAAAAEBAAAAAAIAAAAAAADEAtc9Tjs8vULZ9D1vAR4+/JhpPYf35b5SMjs+c7+tPbjLbr44btc9vJSgvWfwKD510z0+hlGTPbU3Fj7yfAs+3Y/2PWU8wjvmQxy+nBAhPizuyz6Hr5E+AHAWvgPoj77Z4o29sAvHvJ1pXL4MgKU+EY9LPjgR07xJ1HK9DMiePsTlqb7x2Gm9E74UvgOxFb5DXBQ+4lBVvdHTHj3V1nk9prdfPmHu0D6A6yQ9R5P/ve01dT3p5MU9apE5vqdT3D0zn708zYphvr8pGD/YOAu+9VbkPamImj3247E9GyxzPZFYdT5beTy+W2sUPKYWw7tQH7E9co7LvSef37sg2E89XyBuPCs5Lb6j70K+G7Kiu/sPgLsxdHw9Nk0vPW2XY76PQJu+WjcnPjhD+73HKCa+VmavvR0uqz6NT669Pp4nPskWAr7ZJ8S+DVToPHux8T0Vy/A+8862PjFOyj0WFju+fWJ8PkEqyz5mZmO99LhBvqtvw709YEQ944hLveMyBL6oOnQ+R5GFvjMWYzziZhI+zC+evaHrcT4tFUO+G8srvs4HCz7DB2a9TdjDO62CMb6XmpW99nHxudM1Hj+k0Tg8RlcHvm/MkL5CpN4+BLsWPpagXb6viHM+S2xIvra4ir6uAd+9S7hivs7DHD1SkNq9vg/mveXDF74okZK+tlPZvRm4uL5tgG8/EqMUvwOWwb8qb9i+KbWkvmVUfT9NDvA+tfzHvi14MD4eIdO+4dwpPpwE9T50328/ZlZmvwb9gL9PzLM+RbKuv3VP6T5ZK8m8t/OoPudbkr9RPso+mv7MPlIF9T4tP

In [12]:


# ベクトル検索用マッピング
vector_mapping = {
    "settings": {
        "analysis": {
            "tokenizer": {"kuromoji_tokenizer": {"type": "kuromoji_tokenizer"}},
            "analyzer": {
                "kuromoji_analyzer": {
                    "type": "custom",
                    "tokenizer": "kuromoji_tokenizer",
                }
            },
        },
        "index.knn": True
    },
    "mappings": {
        "properties": {
            "永続的識別子": {"type": "keyword"},
            "タイトル": {"type": "text", "analyzer": "kuromoji_analyzer"},
            "著者": {"type": "text", "analyzer": "kuromoji_analyzer"},
            "出版者": {"type": "keyword"},
            "出版日": {"type": "keyword"},
            "出版日（W3CDTF）": {
                "type": "date",
                "format": "yyyy||yyyy-MM||yyyy-MM-dd||strict_date_optional_time||epoch_millis",
            },
            "combined_text": {"type": "text", "analyzer": "kuromoji_analyzer"},
            # ベクトルフィールド
            "title_author_vector": {
                "type": "knn_vector",
                "model_id": "title_author_index",  # KNNモデルのID
            }
        }
    },
}

# 既存のベクトルインデックスを削除（存在する場合）
print("=== ベクトルインデックス作成 ===")
delete_res = requests.delete(f"{BASE_URL}/{VECTOR_INDEX_NAME}", headers=HEADERS)
print(f"既存インデックス削除: {delete_res.status_code}")

# 新しいインデックスを作成
create_res = requests.put(
    f"{BASE_URL}/{VECTOR_INDEX_NAME}", 
    headers=HEADERS, 
    data=json.dumps(vector_mapping)
)
print(f"インデックス作成: {create_res.status_code}")
create_result = create_res.json()
print(create_result)

if create_result.get('acknowledged'):
    print("✅ ベクトル検索用インデックス作成完了")
else:
    print("❌ インデックス作成失敗")

=== ベクトルインデックス作成 ===
既存インデックス削除: 200
インデックス作成: 200
{'acknowledged': True, 'shards_acknowledged': True, 'index': 'ndl-bibliography-vector'}
✅ ベクトル検索用インデックス作成完了


In [ ]:
# OpenSearchへのベクトルデータ投入
print("\n=== OpenSearchへのベクトルデータ投入 ===")

# Bulk APIデータ準備

test_df = df.head(10000).reset_index(drop=True).copy()

# Bulk投入実行
print("Bulk投入実行中...")
chunk_size = 1000
for i in range(0, len(test_df), chunk_size):
    test_df_sclice = test_df.iloc[i:i+chunk_size].reset_index(drop=True).copy()

    vector_actions = test_df_sclice["永続的識別子"].map(
        lambda x: json.dumps(
            {
                "index": {
                    "_index": VECTOR_INDEX_NAME,
                    "_id": x.translate(str.maketrans({":": "-", "/": "-"})),
                }
            }
        )
    )

    # データをJSON形式に変換（ベクトルを含む）
    vector_data_jsons = test_df_sclice.to_json(orient="records", lines=True, force_ascii=False).splitlines()

    vector_jsonls = [
        action + "\n" + data_json + "\n" 
        for action, data_json in zip(vector_actions, vector_data_jsons)
    ]

    chunk = vector_jsonls[i:i+chunk_size]
    print(f"投入中: {i}~{i+len(chunk)}")
    
    res = requests.post(
        f"{BASE_URL}/{VECTOR_INDEX_NAME}/_bulk", 
        headers=HEADERS, 
        data="".join(chunk)
    )
    
    if res.status_code != 200:
        print(f"エラー: {res.status_code} - {res.text}")
        break
    
    bulk_result = res.json()
    if bulk_result.get('errors'):
        print("Bulkエラーが発生:")
        for item in bulk_result.get('items', [])[:3]:  # 最初の3つのエラーを表示
            if 'index' in item and 'error' in item['index']:
                print(f"  {item['index']['error']}")

print("投入完了")

# ドキュメント数確認
count_res = requests.get(f"{BASE_URL}/{VECTOR_INDEX_NAME}/_count", headers=HEADERS)
print(f"投入されたドキュメント数: {count_res.json()['count']}")


=== OpenSearchへのベクトルデータ投入 ===


: 

In [84]:
# ベクトル検索機能の実装
def vector_search(query_text, k=5):
    """
    クエリテキストに基づくベクトル検索
    """
    # クエリをベクトル化
    query_vector = text_to_vector(query_text, model, vector_size)
    
    # OpenSearchでベクトル検索
    search_query = {
        "size": k,
        "_source": ["タイトル", "著者", "出版者", "出版日"],
        "query": {
            "knn": {
                "title_author_vector": {
                    "vector": query_vector.tolist(),
                    "k": k
                }
            }
        }
    }
    
    res = requests.post(
        f"{BASE_URL}/{VECTOR_INDEX_NAME}/_search",
        headers=HEADERS,
        data=json.dumps(search_query)
    )
    
    return res.json()

# ベクトル検索テスト
print("\n=== ベクトル検索テスト ===")

test_queries = [
    "東野圭吾 推理小説",
    "村上春樹 恋愛",
    "夏目漱石 古典文学"
]

for query in test_queries:
    print(f"\n検索クエリ: '{query}'")
    print(vector_search(query, k=5))
    


=== ベクトル検索テスト ===

検索クエリ: '東野圭吾 推理小説'
{'took': 5, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 5, 'relation': 'eq'}, 'max_score': 0.034329545, 'hits': [{'_index': 'ndl-bibliography-vector', '_id': 'info-ndljp-pid-13578533', '_score': 0.034329545, '_source': {'出版者': '博文館', '出版日': '1903 2版', 'タイトル': '相撲大観', '著者': '三木愛花, 山田春塘 共編'}}, {'_index': 'ndl-bibliography-vector', '_id': 'info-ndljp-pid-13436694', '_score': 0.034329545, '_source': {'出版者': '至誠堂', '出版日': '1900', 'タイトル': '四本柱 : 相撲叢談', '著者': '上島永二郎 編'}}, {'_index': 'ndl-bibliography-vector', '_id': 'info-ndljp-pid-13380004', '_score': 0.034329545, '_source': {'出版者': '好文社', '出版日': '大正8', 'タイトル': '舞台のおもかげ', '著者': '安部豊 編'}}, {'_index': 'ndl-bibliography-vector', '_id': 'info-ndljp-pid-13355412', '_score': 0.034329545, '_source': {'出版者': '好文社', '出版日': '大正9', 'タイトル': '舞台のおもかげ', '著者': '安部豊 編'}}, {'_index': 'ndl-bibliography-vector', '_id': 'info-ndljp-pid-13341133', '_sc

In [ ]:
# ハイブリッド検索（キーワード検索 + ベクトル検索）
def hybrid_search(query_text, keyword_weight=0.5, vector_weight=0.5, k=5):
    """
    キーワード検索とベクトル検索を組み合わせたハイブリッド検索
    """
    # クエリをベクトル化
    query_vector = text_to_vector(query_text, model, vector_size)
    
    # ハイブリッド検索クエリ
    search_query = {
        "size": k,
        "_source": ["タイトル", "著者", "出版者", "出版日"],
        "query": {
            "bool": {
                "should": [
                    # キーワード検索部分
                    {
                        "multi_match": {
                            "query": query_text,
                            "fields": ["タイトル^2", "著者^1.5", "combined_text"],
                            "type": "best_fields",
                            # keyword weightはhybrid検索でのスコアに対する重みを指定している。
                            "boost": keyword_weight
                        }
                    },
                    # ベクトル検索部分
                    {
                        "knn": {
                            "title_author_vector": {
                                "vector": query_vector.tolist(),
                                "k": k * 2,  # より多くの候補を取得
                                "boost": vector_weight
                            }
                        }
                    }
                ]
            }
        }
    }
    
    res = requests.post(
        f"{BASE_URL}/{VECTOR_INDEX_NAME}/_search",
        headers=HEADERS,
        data=json.dumps(search_query)
    )
    
    return res.json()

# ハイブリッド検索テスト
print("\n=== ハイブリッド検索テスト ===")

test_query = "東野圭吾 数学 推理"
print(f"検索クエリ: '{test_query}'")
print("=" * 60)

# 3つの検索方法を比較
print("1. キーワード検索のみ")
print("-" * 30)
keyword_result = hybrid_search(test_query, keyword_weight=1.0, vector_weight=0.0, k=3)
if 'hits' in keyword_result and keyword_result['hits']['hits']:
    for i, hit in enumerate(keyword_result['hits']['hits'], 1):
        source = hit['_source']
        score = hit['_score']
        print(f"{i}. スコア: {score:.4f} | {source.get('タイトル', 'N/A')} - {source.get('著者', 'N/A')}")

print("\n2. ベクトル検索のみ")
print("-" * 30)
vector_result = hybrid_search(test_query, keyword_weight=0.0, vector_weight=1.0, k=3)
if 'hits' in vector_result and vector_result['hits']['hits']:
    for i, hit in enumerate(vector_result['hits']['hits'], 1):
        source = hit['_source']
        score = hit['_score']
        print(f"{i}. スコア: {score:.4f} | {source.get('タイトル', 'N/A')} - {source.get('著者', 'N/A')}")

print("\n3. ハイブリッド検索（キーワード50% + ベクトル50%）")
print("-" * 30)
hybrid_result = hybrid_search(test_query, keyword_weight=0.5, vector_weight=0.5, k=3)
if 'hits' in hybrid_result and hybrid_result['hits']['hits']:
    for i, hit in enumerate(hybrid_result['hits']['hits'], 1):
        source = hit['_source']
        score = hit['_score']
        print(f"{i}. スコア: {score:.4f} | {source.get('タイトル', 'N/A')} - {source.get('著者', 'N/A')}")


=== ハイブリッド検索テスト ===
検索クエリ: '東野圭吾 数学 推理'
1. キーワード検索のみ
------------------------------
1. スコア: 9.2359 | 義経伝説推理行 - 荒巻義雄, 合田一道 著
2. スコア: 8.6834 | 古代史を推理する - 邦光史郎 著
3. スコア: 8.6834 | 森村誠一長編推理選集 - None

2. ベクトル検索のみ
------------------------------
1. スコア: 0.0586 | 石川県管内町村元標里粁程表 - 吉田吉三郎 著
2. スコア: 0.0586 | 神奈川県足柄上郡清水村 地目地番反別入図 - None
3. スコア: 0.0586 | 大阪文楽座人形画集 : 日本名物 - 長谷川小信 画

3. ハイブリッド検索（キーワード50% + ベクトル50%）
------------------------------
1. スコア: 4.6179 | 義経伝説推理行 - 荒巻義雄, 合田一道 著
2. スコア: 4.3417 | 古代史を推理する - 邦光史郎 著
3. スコア: 4.3417 | 森村誠一長編推理選集 - None
